<a href="https://colab.research.google.com/github/Ash100/mitraphylline-chemspace/blob/main/mitraphylline_coconut_chemical_space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# A Scalable, Generalizable, and Reproducible Open-Source Framework for Natural-Product Chemical Space Exploration and Analogue Prioritization

### Reference-Guided Case Study on the Oxindole Alkaloid Mitraphylline

---

**Dataset:** COCONUT (COlleCtion of Open Natural prodUcTs), December 2025 snapshot — 715,822 raw entries, deduplicated to **705,603 unique natural products** by InChIKey.

**Reference compound:** Mitraphylline (oxindole alkaloid; COCONUT ID `CNP0159273.3`; InChIKey `JMIAZDVHNCCPDM-DAFCLMLCSA-N`), a pharmacologically studied *Uncaria*/*Mitragyna* alkaloid.

**What this notebook does:**
1. Ingests and cleans the full COCONUT database in a **memory-safe, chunked** manner (validated on a single-core, 4 GB RAM environment — the design generalizes trivially to larger machines).
2. Computes molecular fingerprints (Morgan/ECFP4, r=2, 2048-bit) and Bemis–Murcko scaffolds for every unique compound.
3. Characterizes the **physicochemical and biosynthetic landscape** of the natural-product chemical space (distributions, drug-likeness filters, inter-pathway statistical comparisons).
4. Maps the **global chemical space** via incremental PCA (full dataset) and UMAP (stratified subsample), with quantitative diversity metrics.
5. Performs **reference-guided analogue prioritization** around mitraphylline: full-dataset Tanimoto ranking, statistical placement within the global similarity distribution, Maximum Common Substructure (MCS) scaffold analysis, and separates *true stereoisomers* (Tanimoto = 1.0, non-chiral fingerprint artifact) from *genuinely distinct structural analogues* — extending the framework's applicability beyond simple similarity search into rigorous, publication-grade analogue triage.

**Design principle:** every heavy step below follows a **compute-once, cache, and reuse** pattern — if a cached artifact already exists on disk, the cell loads it instead of recomputing, making the notebook cheap to re-run or extend while remaining byte-for-byte reproducible from a clean environment.

> ⏱️ **Expected runtime from a completely fresh state:** ~35–45 minutes total (cleaning + fingerprinting ≈ 12–15 min; PCA/UMAP ≈ 15–20 min; everything else ≈ 1–2 min), tested on a constrained 1-vCPU / 4 GB RAM machine. On a standard Colab runtime (2+ vCPUs, 12+ GB RAM) this will run considerably faster.



## 0. Environment Setup

Installs pinned dependencies and defines the project directory structure. All paths are relative to `BASE_DIR`, so the notebook works unmodified in Google Colab, a local Jupyter install, or any POSIX environment.


In [ ]:

# ---- Install dependencies (safe to re-run; Colab-friendly) ----
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import rdkit  # noqa
except ImportError:
    pip_install(["rdkit"])

try:
    import umap  # noqa
except ImportError:
    pip_install(["umap-learn"])

for pkg, mod in [("seaborn", "seaborn"), ("scikit-learn", "sklearn"), ("scipy", "scipy")]:
    try:
        __import__(mod)
    except ImportError:
        pip_install([pkg])

print("Environment ready.")


In [ ]:

import os, sys, time, json, gc, pickle, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") if not sys.stdout.isatty() else None
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sps

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Draw, rdFMCS
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9, "font.family": "DejaVu Sans"
})

# ---- Project directory structure ----
BASE_DIR = "./coconut_project"          # change if desired (e.g. a Drive-mounted path)
DATA_DIR      = f"{BASE_DIR}/data"
RAW_DIR       = f"{DATA_DIR}/raw"
CSV_DIR       = f"{BASE_DIR}/results/csv"
FIG_DIR       = f"{BASE_DIR}/results/figures"
FP_DIR        = f"{BASE_DIR}/results/fingerprints"

for d in [DATA_DIR, RAW_DIR, CSV_DIR, FIG_DIR, FP_DIR]:
    os.makedirs(d, exist_ok=True)

RAW_ZIP  = f"{RAW_DIR}/coconut_csv-12-2025.zip"
RAW_CSV  = f"{RAW_DIR}/coconut_csv-12-2025.csv"
CLEANED_CSV = f"{CSV_DIR}/coconut_cleaned.csv"
FP_PATH  = f"{FP_DIR}/morgan2048.dat"
FP_SHAPE_PATH = f"{DATA_DIR}/fp_shape.txt"

MITRAPHYLLINE_INCHIKEY = "JMIAZDVHNCCPDM-DAFCLMLCSA-N"
SEED = 42

print(f"BASE_DIR = {os.path.abspath(BASE_DIR)}")



### Data acquisition

Download the COCONUT database CSV export (`coconut_csv-12-2025.zip`) from the [COCONUT database](https://coconut.naturalproducts.net/) and place it at `RAW_ZIP` (or upload it interactively in Colab with the cell below).


In [ ]:

# ---- Skip entirely if cleaned dataset + fingerprints are already cached ----
_cleaning_cache_ready = (os.path.exists(CLEANED_CSV) and os.path.exists(FP_PATH)
                          and os.path.exists(FP_SHAPE_PATH))

if _cleaning_cache_ready:
    print("[cache hit] Cleaned dataset and fingerprints already exist -- "
          "skipping raw data acquisition entirely.")
else:
    # ---- Colab-friendly upload fallback ----
    if not os.path.exists(RAW_ZIP):
        try:
            from google.colab import files  # type: ignore
            print("RAW_ZIP not found -- please upload coconut_csv-12-2025.zip")
            uploaded = files.upload()
            for fname in uploaded:
                os.rename(fname, RAW_ZIP)
        except ImportError:
            print(f"RAW_ZIP not found at {RAW_ZIP}. "
                  f"Place the COCONUT CSV zip export there before continuing "
                  f"(non-Colab environment: no interactive upload available).")

    if os.path.exists(RAW_ZIP) and not os.path.exists(RAW_CSV):
        with zipfile.ZipFile(RAW_ZIP) as zf:
            zf.extractall(RAW_DIR)
        extracted = [f for f in os.listdir(RAW_DIR) if f.endswith(".csv")]
        if extracted and not os.path.exists(RAW_CSV):
            os.rename(f"{RAW_DIR}/{extracted[0]}", RAW_CSV)

    assert os.path.exists(RAW_CSV), "Raw COCONUT CSV not found -- cannot proceed."
    print(f"Raw COCONUT CSV ready at {RAW_CSV} ({os.path.getsize(RAW_CSV)/1e6:.1f} MB)")



## 1. Data Cleaning, Deduplication & Fingerprinting

COCONUT aggregates entries from many source databases, producing a meaningful fraction of exact structural duplicates (same InChIKey, different source record). We:

1. Stream through the raw CSV **once** to identify, per InChIKey, the row with the richest curation (`annotation_level`) — this is the canonical representative kept for deduplication.
2. Stream through the raw CSV a **second time**, validating and sanitizing each kept SMILES string with RDKit, recomputing a canonical SMILES, computing the Bemis–Murcko scaffold, and generating a 2048-bit Morgan (ECFP4-equivalent, radius=2) fingerprint.
3. Fingerprints are **bit-packed** (`np.packbits`) for an 8× memory reduction and stored as a flat, memory-mapped binary file — allowing constant-RAM random access regardless of dataset size.

Both passes are processed in **fixed-size chunks**, so peak memory stays low and constant regardless of how large the input file is (validated end-to-end on a 1-vCPU / 4 GB RAM machine for ~716K input rows).

**Caching:** if `coconut_cleaned.csv` and the fingerprint store already exist, this whole section is skipped and the cached artifacts are loaded directly.


In [ ]:

FP_NBITS = 2048
FP_RADIUS = 2
CHUNK = 20000

needs_processing = not (os.path.exists(CLEANED_CSV) and os.path.exists(FP_PATH) and os.path.exists(FP_SHAPE_PATH))

if not needs_processing:
    with open(FP_SHAPE_PATH) as f:
        n_rows, n_bytes, n_bits, radius = map(int, f.read().split(","))
    print(f"[cache hit] Using cached cleaned dataset: {n_rows:,} molecules, "
          f"{n_bits}-bit fingerprints (radius={radius}).")
else:
    print("[cache miss] Running full cleaning + fingerprinting pipeline from scratch.")


In [ ]:

# ---- Pass 1: determine deduplication keep-set (by best annotation_level per InChIKey) ----
if needs_processing:
    t0 = time.time()
    best_level = {}
    best_row_idx = {}
    n_total = 0
    for chunk in pd.read_csv(RAW_CSV, chunksize=CHUNK, low_memory=False,
                              usecols=['identifier', 'canonical_smiles',
                                       'standard_inchi_key', 'annotation_level']):
        for local_i, row in enumerate(chunk.itertuples(index=False)):
            global_i = n_total + local_i
            key, smi = row.standard_inchi_key, row.canonical_smiles
            if not isinstance(key, str) or not isinstance(smi, str) or len(smi) == 0:
                continue
            lvl = row.annotation_level
            if key not in best_level or lvl > best_level[key]:
                best_level[key] = lvl
                best_row_idx[key] = global_i
        n_total += len(chunk)

    keep_indices = set(best_row_idx.values())
    print(f"Pass 1 done in {time.time()-t0:.1f}s: {n_total:,} raw rows -> "
          f"{len(keep_indices):,} unique InChIKeys kept "
          f"({100*(1 - len(keep_indices)/n_total):.2f}% duplicates removed).")


In [ ]:

# ---- Pass 2: validate SMILES, compute scaffolds + fingerprints for kept rows ----
if needs_processing:
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)
    n_keep = len(keep_indices)
    n_bytes_per_fp = FP_NBITS // 8
    fp_memmap = np.memmap(FP_PATH, dtype=np.uint8, mode='w+', shape=(n_keep, n_bytes_per_fp))

    out_rows, write_header = [], True
    global_i, kept_counter, n_parse_fail = -1, 0, 0
    t0 = time.time()

    for chunk in pd.read_csv(RAW_CSV, chunksize=CHUNK, low_memory=False):
        smiles_arr = chunk['canonical_smiles'].to_numpy()
        for local_pos in range(len(chunk)):
            global_i += 1
            if global_i not in keep_indices:
                continue
            smi = smiles_arr[local_pos]
            mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
            if mol is None:
                n_parse_fail += 1
                continue
            try:
                recanon_smiles = Chem.MolToSmiles(mol)
                scaffold = MurckoScaffold.GetScaffoldForMol(mol)
                scaffold_smiles = Chem.MolToSmiles(scaffold) if scaffold is not None else ""
                bitvec = mfpgen.GetFingerprint(mol)
                arr = np.frombuffer(bytes(bitvec.ToBitString(), 'ascii'), dtype=np.uint8) - ord('0')
                packed = np.packbits(arr.astype(np.uint8))
            except Exception:
                n_parse_fail += 1
                continue

            fp_memmap[kept_counter] = packed
            row = chunk.iloc[local_pos]
            rec = row.to_dict()
            rec['recanon_smiles'] = recanon_smiles
            rec['murcko_scaffold_recomputed'] = scaffold_smiles
            rec['fp_row_index'] = kept_counter
            out_rows.append(rec)
            kept_counter += 1

        if len(out_rows) >= 100000:
            pd.DataFrame(out_rows).to_csv(CLEANED_CSV, mode='a', header=write_header, index=False)
            write_header = False
            out_rows = []

    if out_rows:
        pd.DataFrame(out_rows).to_csv(CLEANED_CSV, mode='a', header=write_header, index=False)
    fp_memmap.flush()

    with open(FP_SHAPE_PATH, "w") as f:
        f.write(f"{kept_counter},{n_bytes_per_fp},{FP_NBITS},{FP_RADIUS}")

    print(f"Pass 2 done in {time.time()-t0:.1f}s: {kept_counter:,} molecules processed, "
          f"{n_parse_fail} parse failures ({100*n_parse_fail/n_keep:.4f}%).")

    n_rows, n_bytes, n_bits, radius = kept_counter, n_bytes_per_fp, FP_NBITS, FP_RADIUS


In [ ]:

# ---- Load fingerprint store as a memory-mapped array (constant RAM regardless of size) ----
def load_fp_memmap(path, n_rows, n_bytes):
    return np.memmap(path, dtype=np.uint8, mode='r', shape=(n_rows, n_bytes))

fps = load_fp_memmap(FP_PATH, n_rows, n_bytes)
print(f"Fingerprint store: {fps.shape[0]:,} molecules x {n_bits}-bit Morgan/ECFP4 (radius={radius})")


Fingerprint store: 705,603 molecules x 2048-bit Morgan/ECFP4 (radius=2)



## 2. Fast Bit-Packed Tanimoto Similarity

Rather than depending on RDKit's per-pair Python-level similarity calls (slow at scale) or a compiled extension, we implement Tanimoto similarity as **vectorized numpy operations directly on packed bits**, using a precomputed 256-entry popcount lookup table. This makes a "1-vs-all" similarity search against ~700K compounds run in **under 2 seconds**, entirely in pure numpy with no additional dependencies.


In [ ]:

_POPCOUNT_TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint16)

def popcount_bytes(arr):
    '''Row-wise popcount of a (n, n_bytes) uint8 array -> (n,) int array.'''
    return _POPCOUNT_TABLE[arr].sum(axis=-1)

def tanimoto_one_vs_many(query_packed, others_packed):
    '''query_packed: (n_bytes,) uint8; others_packed: (n, n_bytes) uint8 -> (n,) similarities.'''
    query_popcount = _POPCOUNT_TABLE[query_packed].sum()
    others_popcount = popcount_bytes(others_packed)
    intersection = popcount_bytes(np.bitwise_and(others_packed, query_packed))
    union = query_popcount + others_popcount - intersection
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(union > 0, intersection / union, 0.0)

def tanimoto_pairwise_batch(fp_a, fp_b):
    '''Full pairwise Tanimoto matrix between two (small) sets -- O(na*nb), for diversity estimation.'''
    pa = popcount_bytes(fp_a)[:, None]
    pb = popcount_bytes(fp_b)[None, :]
    na, nb = fp_a.shape[0], fp_b.shape[0]
    inter = np.zeros((na, nb), dtype=np.int32)
    for j in range(nb):
        inter[:, j] = popcount_bytes(np.bitwise_and(fp_a, fp_b[j]))
    union = pa + pb - inter
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(union > 0, inter / union, 0.0)

print("Tanimoto utilities ready.")



## 3. Physicochemical & Biosynthetic Landscape of the Cleaned COCONUT Set

COCONUT ships with a rich set of precomputed descriptors (molecular weight, ALogP, TPSA, H-bond donors/acceptors, QED, fraction Csp3, ring counts, NP-likeness score, NPClassifier pathway/superclass/class). We use these directly rather than recomputing them, and add:

- **Drug-likeness filter compliance** (Lipinski Ro5, Veber, Ghose).
- **Kruskal–Wallis tests** (non-parametric ANOVA) comparing property distributions across the top NPClassifier biosynthetic pathways, with **epsilon-squared effect sizes** and Bonferroni-corrected significance.
- **Spearman correlation structure** among all numeric descriptors.
- **Scaffold diversity metrics** (Shannon entropy, singleton fraction) from Bemis–Murcko scaffolds.

**Caching:** if `table1_descriptive_stats.csv` already exists, this section loads and displays cached results instead of recomputing.


In [ ]:

NUMERIC_COLS = [
    "molecular_weight", "alogp", "topological_polar_surface_area",
    "rotatable_bond_count", "hydrogen_bond_acceptors", "hydrogen_bond_donors",
    "aromatic_rings_count", "qed_drug_likeliness", "fractioncsp3",
    "number_of_minimal_rings", "np_likeness"
]
TITLES = {
    "molecular_weight": "Molecular Weight (Da)", "alogp": "ALogP",
    "topological_polar_surface_area": "TPSA (\u00c5\u00b2)",
    "rotatable_bond_count": "Rotatable Bonds", "hydrogen_bond_acceptors": "H-Bond Acceptors",
    "hydrogen_bond_donors": "H-Bond Donors", "aromatic_rings_count": "Aromatic Rings",
    "qed_drug_likeliness": "QED", "fractioncsp3": "Fraction Csp3",
    "number_of_minimal_rings": "Ring Count", "np_likeness": "NP-likeness Score",
}

TABLE1_PATH = f"{CSV_DIR}/table1_descriptive_stats.csv"
COMPLIANCE_PATH = f"{CSV_DIR}/lipinski_veber_compliance.csv"
KW_PATH = f"{CSV_DIR}/kruskal_wallis_pathway_tests.csv"
PATHWAY_MEDIANS_PATH = f"{CSV_DIR}/np_pathway_property_medians.csv"
SCAFFOLD_DIV_PATH = f"{CSV_DIR}/scaffold_diversity_summary.csv"

usecols_stats = NUMERIC_COLS + [
    "lipinski_rule_of_five_violations", "np_classifier_pathway", "np_classifier_superclass",
    "contains_sugar", "contains_ring_sugars", "contains_linear_sugars",
    "murcko_scaffold_recomputed", "np_classifier_is_glycoside", "identifier"
]
df_stats = pd.read_csv(CLEANED_CSV, usecols=usecols_stats, low_memory=False)
n = len(df_stats)
print(f"Loaded {n:,} rows for descriptive analysis.")


Loaded 705,603 rows for descriptive analysis.


In [ ]:

if os.path.exists(TABLE1_PATH):
    desc = pd.read_csv(TABLE1_PATH, index_col=0)
    print("[cache hit] Loaded Table 1 (descriptive statistics).")
else:
    desc = df_stats[NUMERIC_COLS].describe(percentiles=[0.25, 0.5, 0.75]).T
    desc["IQR"] = desc["75%"] - desc["25%"]
    desc["skew"] = df_stats[NUMERIC_COLS].skew()
    desc.to_csv(TABLE1_PATH)
    print("Computed and cached Table 1.")
desc


In [ ]:

if os.path.exists(COMPLIANCE_PATH):
    compliance = pd.read_csv(COMPLIANCE_PATH)
else:
    lipinski_pass = (df_stats["lipinski_rule_of_five_violations"] <= 1)
    veber_pass = (df_stats["rotatable_bond_count"] <= 10) & (df_stats["topological_polar_surface_area"] <= 140)
    ghose_pass = df_stats["molecular_weight"].between(160, 480) & df_stats["alogp"].between(-0.4, 5.6)
    compliance = pd.DataFrame({
        "rule": ["Lipinski Ro5 (<=1 violation)", "Veber", "Ghose (MW & LogP window)"],
        "n_pass": [lipinski_pass.sum(), veber_pass.sum(), ghose_pass.sum()],
        "pct_pass": [100*lipinski_pass.mean(), 100*veber_pass.mean(), 100*ghose_pass.mean()]
    })
    compliance.to_csv(COMPLIANCE_PATH, index=False)
compliance


In [ ]:

df_stats["np_classifier_pathway"] = df_stats["np_classifier_pathway"].fillna("Unclassified")
top_pathways = df_stats["np_classifier_pathway"].value_counts().head(8).index.tolist()
sub = df_stats[df_stats["np_classifier_pathway"].isin(top_pathways)]

if os.path.exists(KW_PATH):
    kw_df = pd.read_csv(KW_PATH)
else:
    kw_results = []
    for col in NUMERIC_COLS:
        groups = [g[col].dropna().values for _, g in sub.groupby("np_classifier_pathway")]
        groups = [g for g in groups if len(g) > 5]
        n_total_kw = sum(len(g) for g in groups)
        if len(groups) >= 2:
            h, p = sps.kruskal(*groups)
            eps_sq = h / ((n_total_kw**2 - 1) / (n_total_kw + 1)) if n_total_kw > 1 else np.nan
            kw_results.append({"property": col, "H_statistic": h, "p_value": p,
                                "n_groups": len(groups), "n_total": n_total_kw,
                                "epsilon_squared_effect_size": eps_sq})
    kw_df = pd.DataFrame(kw_results)
    kw_df["significant_bonferroni_0.05"] = kw_df["p_value"] < (0.05 / len(kw_df))
    kw_df.to_csv(KW_PATH, index=False)

print("Kruskal-Wallis tests: property distributions across NPClassifier pathways")
kw_df



**Interpretation:** every physicochemical descriptor differs significantly (Bonferroni-corrected p < 0.05) across biosynthetic pathways, confirming that NPClassifier pathway assignment captures real, systematic physicochemical structure in the data rather than being an arbitrary label — a useful sanity check before using pathway as a stratification variable for chemical-space sampling (Section 4).


In [ ]:

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()
for ax, col in zip(axes, NUMERIC_COLS):
    vals = df_stats[col].dropna()
    lo, hi = vals.quantile([0.005, 0.995])
    sns.histplot(vals.clip(lo, hi), bins=60, ax=ax, color="#2b6f77", edgecolor=None, kde=True)
    ax.set_title(TITLES.get(col, col)); ax.set_xlabel(""); ax.set_ylabel("Count")
axes[-1].axis("off")
plt.suptitle(f"Physicochemical Property Distributions -- COCONUT (n={n:,} unique natural products)",
             y=1.00, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig1_property_distributions.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

corr = df_stats[NUMERIC_COLS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8, "label": "Spearman \u03c1"}, ax=ax,
            annot_kws={"size": 7})
ax.set_title("Spearman Correlation Among Physicochemical Descriptors", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig2_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
plot_cols = ["molecular_weight", "alogp", "np_likeness", "qed_drug_likeliness"]
for ax, col in zip(axes.flatten(), plot_cols):
    order = sub.groupby("np_classifier_pathway")[col].median().sort_values().index
    sns.boxplot(data=sub, x="np_classifier_pathway", y=col, order=order, ax=ax,
                hue="np_classifier_pathway", palette="viridis", showfliers=False, legend=False)
    ax.set_title(TITLES.get(col, col)); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
plt.suptitle("Property Distributions Across NPClassifier Biosynthetic Pathways", y=1.01,
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig3_pathway_boxplots.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

if os.path.exists(SCAFFOLD_DIV_PATH):
    scaffold_summary = pd.read_csv(SCAFFOLD_DIV_PATH)
else:
    scaffold_counts = df_stats["murcko_scaffold_recomputed"].fillna("").value_counts()
    scaffold_counts_nonempty = scaffold_counts[scaffold_counts.index != ""]
    p = scaffold_counts_nonempty.values / scaffold_counts_nonempty.sum()
    shannon = -np.sum(p * np.log(p))
    n_singleton = (scaffold_counts_nonempty == 1).sum()
    scaffold_summary = pd.DataFrame([{
        "n_compounds": n, "n_unique_scaffolds": len(scaffold_counts_nonempty),
        "scaffold_to_compound_ratio": len(scaffold_counts_nonempty)/n,
        "shannon_entropy": shannon, "n_singleton_scaffolds": n_singleton,
        "pct_singleton_scaffolds": 100*n_singleton/len(scaffold_counts_nonempty)
    }])
    scaffold_summary.to_csv(SCAFFOLD_DIV_PATH, index=False)
scaffold_summary



## 4. Chemical Space Mapping (PCA + UMAP)

**Design rationale for large-scale visualization:**

1. **Incremental PCA**, fit in small batches directly on unpacked fingerprint bits, is run on the **entire cleaned dataset** (no subsampling) — this never materializes the full dense (n × 2048) matrix in memory, and captures genuine global variance structure.
2. **UMAP**, which scales much worse than PCA with dataset size on constrained hardware, is instead computed on a **stratified representative subsample** (stratified by NPClassifier pathway, so all major biosynthetic classes remain proportionally represented), operating on the top-30 PCA components rather than raw 2048-dim bits. This PCA-then-UMAP pattern is standard practice for visualizing large chemical libraries and is documented transparently here as a compute-budget decision — it does **not** affect the full-dataset PCA statistics reported above.
3. **Checkpointing:** the incremental PCA fit and transform loops persist their progress every 50,000 rows, so this section can be safely interrupted and resumed without recomputation — important when processing large libraries on modest hardware (validated end-to-end on a 1-vCPU / 4 GB RAM machine, where an unchecked pipeline was empirically observed to be killed by the OS OOM killer without checkpointing).

**Caching:** if `pca_variance_explained.csv` and `umap_embedding_subsample.csv` already exist, this section loads them directly.


In [ ]:

PCA_VAR_PATH = f"{CSV_DIR}/pca_variance_explained.csv"
PC10_PATH = f"{CSV_DIR}/full_dataset_pc10_coordinates.csv"
UMAP_PATH = f"{CSV_DIR}/umap_embedding_subsample.csv"
DIVERSITY_PATH = f"{CSV_DIR}/diversity_metrics.csv"
IPCA_CKPT = f"{DATA_DIR}/ipca_checkpoint.pkl"
IPCA_PROGRESS = f"{DATA_DIR}/ipca_progress.txt"
PC10_CKPT = f"{DATA_DIR}/pc10_partial.npy"
PC10_PROGRESS = f"{DATA_DIR}/pc10_progress.txt"

N_COMPONENTS = 30
BATCH = 2000
CKPT_EVERY = 50000

meta = pd.read_csv(CLEANED_CSV, usecols=[
    "fp_row_index", "identifier", "np_classifier_pathway", "np_classifier_superclass",
    "molecular_weight", "alogp", "np_likeness", "qed_drug_likeliness", "contains_sugar"
], low_memory=False).sort_values("fp_row_index").reset_index(drop=True)
assert len(meta) == n_rows

need_pca = not os.path.exists(PCA_VAR_PATH)
print(f"PCA {'needs computing' if need_pca else '[cache hit] found'} at {PCA_VAR_PATH}")


PCA [cache hit] found at ./coconut_project/results/csv/pca_variance_explained.csv


In [ ]:

from sklearn.decomposition import IncrementalPCA

if need_pca:
    t0 = time.time()
    if os.path.exists(IPCA_CKPT) and os.path.exists(IPCA_PROGRESS):
        with open(IPCA_CKPT, "rb") as f:
            ipca = pickle.load(f)
        with open(IPCA_PROGRESS) as f:
            resume_start = int(f.read().strip())
        print(f"Resuming IncrementalPCA fit from checkpoint at row {resume_start:,}")
    else:
        ipca = IncrementalPCA(n_components=N_COMPONENTS, batch_size=BATCH)
        resume_start = 0
        print("Fitting IncrementalPCA from scratch (checkpointed every 50,000 rows)...")

    since_ckpt = 0
    for start in range(resume_start, n_rows, BATCH):
        end = min(start + BATCH, n_rows)
        packed_batch = np.array(fps[start:end])
        unpacked = np.unpackbits(packed_batch, axis=1).astype(np.float32)
        del packed_batch
        if unpacked.shape[0] >= N_COMPONENTS:
            ipca.partial_fit(unpacked)
        del unpacked
        since_ckpt += (end - start)
        if since_ckpt >= CKPT_EVERY:
            gc.collect()
            with open(IPCA_CKPT, "wb") as f: pickle.dump(ipca, f)
            with open(IPCA_PROGRESS, "w") as f: f.write(str(end))
            since_ckpt = 0
            print(f"  checkpoint at {end:,}/{n_rows:,} rows, elapsed {time.time()-t0:.1f}s")

    gc.collect()
    with open(IPCA_CKPT, "wb") as f: pickle.dump(ipca, f)
    with open(IPCA_PROGRESS, "w") as f: f.write(str(n_rows))

    var_ratio = ipca.explained_variance_ratio_
    pd.DataFrame({
        "component": np.arange(1, N_COMPONENTS + 1),
        "explained_variance_ratio": var_ratio,
        "cumulative_variance": np.cumsum(var_ratio)
    }).to_csv(PCA_VAR_PATH, index=False)
    print(f"PCA fit done in {time.time()-t0:.1f}s. "
          f"Top-{N_COMPONENTS} PCs explain {np.cumsum(var_ratio)[-1]*100:.1f}% of variance.")
else:
    with open(IPCA_CKPT, "rb") as f:
        ipca = pickle.load(f)
    var_ratio = ipca.explained_variance_ratio_
    print("[cache hit] Loaded fitted IncrementalPCA model.")

pca_var_df = pd.read_csv(PCA_VAR_PATH)
pca_var_df.head(10)


In [ ]:

if not os.path.exists(PC10_PATH):
    if os.path.exists(PC10_CKPT) and os.path.exists(PC10_PROGRESS):
        pc10 = np.load(PC10_CKPT)
        with open(PC10_PROGRESS) as f:
            t_resume = int(f.read().strip())
    else:
        pc10 = np.zeros((n_rows, 10), dtype=np.float32)
        t_resume = 0

    t0 = time.time()
    since_ckpt = 0
    for start in range(t_resume, n_rows, BATCH):
        end = min(start + BATCH, n_rows)
        packed_batch = np.array(fps[start:end])
        unpacked = np.unpackbits(packed_batch, axis=1).astype(np.float32)
        pc10[start:end] = ipca.transform(unpacked)[:, :10]
        del packed_batch, unpacked
        since_ckpt += (end - start)
        if since_ckpt >= CKPT_EVERY:
            gc.collect()
            np.save(PC10_CKPT, pc10)
            with open(PC10_PROGRESS, "w") as f: f.write(str(end))
            since_ckpt = 0
            print(f"  transform checkpoint at {end:,}/{n_rows:,} rows, elapsed {time.time()-t0:.1f}s")

    meta_pc = meta.copy()
    for i in range(10):
        meta_pc[f"PC{i+1}"] = pc10[:, i]
    meta_pc.to_csv(PC10_PATH, index=False)
    print(f"Full-dataset 10-PC coordinates saved ({time.time()-t0:.1f}s).")
else:
    print("[cache hit] Loaded full-dataset PC coordinates.")
meta_pc = pd.read_csv(PC10_PATH, low_memory=False)


In [ ]:

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(range(1, 21), var_ratio[:20] * 100, color="#4c72b0", alpha=0.8, label="Individual")
ax2 = ax.twinx()
ax2.plot(range(1, 21), np.cumsum(var_ratio[:20]) * 100, color="#c44e52", marker="o", markersize=3,
         label="Cumulative")
ax.set_xlabel("Principal Component"); ax.set_ylabel("Explained Variance (%)")
ax2.set_ylabel("Cumulative Variance (%)")
ax.set_title(f"PCA Scree Plot (Morgan Fingerprint Space, full n={n_rows:,})")
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, loc="center right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig5_pca_scree.png", dpi=300, bbox_inches="tight")
plt.show()



### Stratified subsample + UMAP embedding

30,000 compounds sampled proportionally across NPClassifier pathways (fixed seed for reproducibility), projected from the fitted 30-component PCA space into 2D via UMAP.


In [ ]:

import umap

SUBSAMPLE_N = 30000
rng = np.random.default_rng(SEED)

if not os.path.exists(UMAP_PATH):
    meta["np_classifier_pathway"] = meta["np_classifier_pathway"].fillna("Unclassified")
    strata = meta["np_classifier_pathway"]
    frac = min(1.0, SUBSAMPLE_N / n_rows)
    sub_idx = (meta.groupby(strata, group_keys=False)
               .apply(lambda g: g.sample(frac=frac, random_state=SEED)).index.to_numpy())
    if len(sub_idx) > SUBSAMPLE_N:
        sub_idx = rng.choice(sub_idx, size=SUBSAMPLE_N, replace=False)
    sub_idx = np.sort(sub_idx)

    t0 = time.time()
    sub_packed = np.array(fps[sub_idx])
    sub_unpacked = np.unpackbits(sub_packed, axis=1).astype(np.float32)
    sub_pc = ipca.transform(sub_unpacked)
    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, metric="euclidean",
                         random_state=SEED, n_components=2, low_memory=True)
    embedding = reducer.fit_transform(sub_pc)
    print(f"UMAP embedding of {len(sub_idx):,} compounds done in {time.time()-t0:.1f}s")

    umap_df = meta.iloc[sub_idx].copy()
    umap_df["UMAP1"], umap_df["UMAP2"] = embedding[:, 0], embedding[:, 1]
    umap_df.to_csv(UMAP_PATH, index=False)
else:
    print("[cache hit] Loaded UMAP embedding.")
umap_df = pd.read_csv(UMAP_PATH, low_memory=False)
umap_df.head()


In [ ]:

top_pathways_umap = umap_df["np_classifier_pathway"].fillna("Unclassified").value_counts().head(7).index.tolist()
umap_df["pathway_plot"] = umap_df["np_classifier_pathway"].fillna("Unclassified").where(
    umap_df["np_classifier_pathway"].isin(top_pathways_umap), "Other")

fig, ax = plt.subplots(figsize=(8, 7))
sns.scatterplot(data=umap_df, x="UMAP1", y="UMAP2", hue="pathway_plot", s=6, alpha=0.6, ax=ax,
                 palette="tab10", linewidth=0)
ax.set_title(f"UMAP Projection of COCONUT Chemical Space\n"
             f"(n={len(umap_df):,} stratified subsample, colored by biosynthetic pathway)",
             fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, title="Pathway", title_fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig6_umap_chemical_space.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col, cmap, label in zip(
        axes, ["molecular_weight", "alogp", "np_likeness"],
        ["viridis", "coolwarm", "plasma"],
        ["Molecular Weight (Da)", "ALogP", "NP-likeness score"]):
    vals = umap_df[col]
    lo, hi = vals.quantile([0.02, 0.98])
    sc = ax.scatter(umap_df["UMAP1"], umap_df["UMAP2"], c=vals.clip(lo, hi), cmap=cmap,
                     s=5, alpha=0.7, linewidth=0)
    plt.colorbar(sc, ax=ax, label=label, shrink=0.8)
    ax.set_title(label); ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
plt.suptitle("Chemical Space Colored by Physicochemical Properties", y=1.03,
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig7_umap_properties.png", dpi=300, bbox_inches="tight")
plt.show()



### Diversity metrics

Mean pairwise Tanimoto similarity (and its complement, 1−T, as a diversity index) estimated from a random sample of 3,000 compounds (quadratic cost makes full all-vs-all computation on 700K compounds impractical; this sample size gives a stable estimate with ~4.5M pairs).


In [ ]:

if not os.path.exists(DIVERSITY_PATH):
    DIV_SAMPLE = 3000
    div_idx = np.sort(rng.choice(n_rows, size=min(DIV_SAMPLE, n_rows), replace=False))
    div_fps = np.array(fps[div_idx])
    t0 = time.time()
    sim_matrix = tanimoto_pairwise_batch(div_fps, div_fps)
    iu = np.triu_indices_from(sim_matrix, k=1)
    pairwise_sims = sim_matrix[iu]
    diversity_summary = pd.DataFrame([{
        "sample_size": len(div_idx), "n_pairs": len(pairwise_sims),
        "mean_pairwise_tanimoto": pairwise_sims.mean(),
        "median_pairwise_tanimoto": np.median(pairwise_sims),
        "std_pairwise_tanimoto": pairwise_sims.std(),
        "mean_pairwise_diversity_1_minus_T": 1 - pairwise_sims.mean(),
    }])
    diversity_summary.to_csv(DIVERSITY_PATH, index=False)
    print(f"Diversity metrics computed in {time.time()-t0:.1f}s.")
else:
    print("[cache hit] Loaded diversity metrics.")
pd.read_csv(DIVERSITY_PATH)



## 5. Reference-Guided Analogue Prioritization: Mitraphylline

This section demonstrates the framework's core intended use case: given a single reference natural product, rank the **entire cleaned COCONUT library** by structural similarity, statistically contextualize the hits, and separate genuinely novel structural analogues from trivial stereoisomeric duplicates.

**Reference compound:** Mitraphylline — an oxindole alkaloid from *Uncaria*/*Mitragyna* spp. (InChIKey `JMIAZDVHNCCPDM-DAFCLMLCSA-N`).

**Method:**
1. Locate mitraphylline's canonical (deduplicated) database entry.
2. Compute Tanimoto similarity to **every one of the ~700K unique compounds** in the cleaned set — a full, non-subsampled comparison, made tractable by the vectorized bit-packed popcount approach from Section 2 (runs in ~2 seconds).
3. Rank and statistically contextualize the top hits (percentile rank and z-score within the *complete* similarity distribution).
4. Compute Maximum Common Substructure (MCS) overlap between mitraphylline and each hit, to quantify shared scaffold extent independent of the bit-vector similarity metric.
5. **Critically**, separate two tiers of results:
   - **Tier A — Nearest neighbors (unfiltered):** may include compounds with Tanimoto = 1.0. As shown below, these are *not duplicates or artifacts* — they are mitraphylline's well-documented natural diastereomers (pteropodine, isopteropodine, speciophylline, uncarine A/F, isomitraphylline), which share identical 2D constitutional structure and differ only in stereocenters. Standard circular fingerprints (Morgan/ECFP) are **not stereochemistry-aware**, so this tier is expected and, in fact, functions as a positive control confirming COCONUT correctly captures mitraphylline's full known stereoisomeric family.
   - **Tier B — Structurally distinct analogues (Tanimoto < 1.0):** obtained by excluding exact 2D matches, this tier surfaces genuinely different scaffolds/substitution patterns for analogue prioritization — the practically useful output for medicinal-chemistry follow-up.


In [ ]:

meta_full = pd.read_csv(CLEANED_CSV, low_memory=False).sort_values("fp_row_index").reset_index(drop=True)
assert len(meta_full) == n_rows

ref_rows = meta_full[meta_full["standard_inchi_key"] == MITRAPHYLLINE_INCHIKEY]
assert len(ref_rows) > 0, "Mitraphylline InChIKey not found in cleaned dataset!"
ref_row = ref_rows.iloc[0]
ref_fp_idx = int(ref_row["fp_row_index"])
ref_packed = np.array(fps[ref_fp_idx])
ref_smiles = ref_row["recanon_smiles"]
ref_mol = Chem.MolFromSmiles(ref_smiles)

print(f"Reference compound: {ref_row['identifier']} | {ref_row.get('name', 'Mitraphylline')}")
print(f"SMILES: {ref_smiles}")
Draw.MolToImage(ref_mol, size=(350, 300))


In [ ]:

t0 = time.time()
all_packed = np.array(fps)
sims = tanimoto_one_vs_many(ref_packed, all_packed)
print(f"Computed {n_rows:,} similarities to mitraphylline in {time.time()-t0:.1f}s")

meta_full["tanimoto_to_mitraphylline"] = sims
non_self = meta_full[meta_full["standard_inchi_key"] != MITRAPHYLLINE_INCHIKEY].copy()
ranked_all = non_self.sort_values("tanimoto_to_mitraphylline", ascending=False).copy()

# Tier A: raw top-10 by similarity
top10 = ranked_all.head(10).copy().reset_index(drop=True)
top10.insert(0, "rank", np.arange(1, len(top10) + 1))

n_exact_2d_duplicates = int((ranked_all["tanimoto_to_mitraphylline"] >= 0.9999).sum())
print(f"\n{n_exact_2d_duplicates} compounds share Tanimoto=1.0 with mitraphylline "
      f"(expected: non-chiral fingerprint cannot distinguish mitraphylline's diastereomers).")

# Tier B: top-10 excluding exact 2D duplicates (genuine structural analogues)
distinct = ranked_all[ranked_all["tanimoto_to_mitraphylline"] < 0.9999].copy()
top10_distinct = distinct.head(10).copy().reset_index(drop=True)
top10_distinct.insert(0, "rank", np.arange(1, len(top10_distinct) + 1))
print(f"Tier B top hit (structurally distinct): {top10_distinct.iloc[0]['identifier']} "
      f"@ T={top10_distinct.iloc[0]['tanimoto_to_mitraphylline']:.3f}")


In [ ]:

sim_mean, sim_std = sims.mean(), sims.std()

def add_stats(df):
    df = df.copy()
    df["percentile_in_full_distribution"] = [np.mean(sims <= s) * 100 for s in df["tanimoto_to_mitraphylline"]]
    df["z_score"] = [(s - sim_mean) / sim_std for s in df["tanimoto_to_mitraphylline"]]
    return df

top10 = add_stats(top10)
top10_distinct = add_stats(top10_distinct)

dist_summary = pd.DataFrame([{
    "n_compounds_compared": n_rows, "mean_tanimoto": sim_mean, "median_tanimoto": np.median(sims),
    "std_tanimoto": sim_std, "max_tanimoto_excl_self": non_self["tanimoto_to_mitraphylline"].max(),
    "n_exact_2d_stereoisomers": n_exact_2d_duplicates,
    "top_distinct_analogue_tanimoto": top10_distinct.iloc[0]["tanimoto_to_mitraphylline"],
    "p99_tanimoto": np.percentile(sims, 99), "p99.9_tanimoto": np.percentile(sims, 99.9),
}])
dist_summary.to_csv(f"{CSV_DIR}/mitraphylline_similarity_distribution_summary.csv", index=False)
dist_summary



**Statistical interpretation:** with a mean pairwise Tanimoto of ~0.12 across the full library, mitraphylline's Tier-B analogues (T ≈ 0.80–0.82) sit at approximately the **99.997th percentile** of the entire similarity distribution — roughly 18–19 standard deviations above the library-wide mean. This is a much stronger and more interpretable statement than "Tanimoto > 0.8" in isolation, since it is calibrated against the actual empirical background of this specific chemical library rather than an arbitrary universal threshold.


In [ ]:

def compute_mcs(df):
    df = df.copy()
    mcs_frac_ref, mcs_smarts_list, mcs_natoms = [], [], []
    for smi in df["recanon_smiles"]:
        mol = Chem.MolFromSmiles(smi)
        try:
            mcs_result = rdFMCS.FindMCS([ref_mol, mol], timeout=10,
                                         bondCompare=rdFMCS.BondCompare.CompareOrder,
                                         atomCompare=rdFMCS.AtomCompare.CompareElements)
            mcs_natoms.append(mcs_result.numAtoms)
            mcs_frac_ref.append(mcs_result.numAtoms / ref_mol.GetNumAtoms())
            mcs_smarts_list.append(mcs_result.smartsString)
        except Exception:
            mcs_natoms.append(np.nan); mcs_frac_ref.append(np.nan); mcs_smarts_list.append("")
    df["mcs_num_atoms"] = mcs_natoms
    df["mcs_fraction_of_mitraphylline"] = mcs_frac_ref
    df["mcs_smarts"] = mcs_smarts_list
    return df

top10 = compute_mcs(top10)
top10_distinct = compute_mcs(top10_distinct)

export_cols = ["rank", "identifier", "name", "recanon_smiles", "molecular_formula",
               "tanimoto_to_mitraphylline", "percentile_in_full_distribution", "z_score",
               "mcs_num_atoms", "mcs_fraction_of_mitraphylline",
               "molecular_weight", "alogp", "topological_polar_surface_area",
               "rotatable_bond_count", "hydrogen_bond_acceptors", "hydrogen_bond_donors",
               "qed_drug_likeliness", "np_likeness", "np_classifier_pathway",
               "np_classifier_superclass", "np_classifier_class", "organisms"]
top10[export_cols].to_csv(f"{CSV_DIR}/mitraphylline_top10_neighbors.csv", index=False)
top10_distinct[export_cols].to_csv(f"{CSV_DIR}/mitraphylline_top10_distinct_analogues.csv", index=False)

print("Tier A (nearest neighbors, incl. stereoisomers):")
display(top10[["rank", "identifier", "name", "tanimoto_to_mitraphylline",
               "percentile_in_full_distribution", "z_score"]])
print("\nTier B (structurally distinct analogues):")
display(top10_distinct[["rank", "identifier", "name", "tanimoto_to_mitraphylline",
                        "percentile_in_full_distribution", "z_score"]])


In [ ]:

def make_structure_grid(df, out_path):
    mols_for_grid = [ref_mol] + [Chem.MolFromSmiles(s) for s in df["recanon_smiles"]]
    legends = [f"Mitraphylline (reference)\n{ref_row['identifier']}"] + [
        f"Rank {r} | T={t:.3f}\n{ident}"
        for r, t, ident in zip(df["rank"], df["tanimoto_to_mitraphylline"], df["identifier"])
    ]
    img = Draw.MolsToGridImage(mols_for_grid, molsPerRow=4, subImgSize=(320, 300),
                                legends=legends, useSVG=False, returnPNG=False)
    if hasattr(img, "save"):
        img.save(out_path)
    else:
        # Running inside a Jupyter kernel: RDKit may return an IPython display
        # Image object instead of a PIL Image -- extract raw PNG bytes instead.
        data = getattr(img, "data", None)
        if data is None:
            data = img._repr_png_()
        with open(out_path, "wb") as f:
            f.write(data)
    return img

img_a = make_structure_grid(top10, f"{FIG_DIR}/fig8_mitraphylline_neighbors_grid.png")
img_a


**Tier A** (above): mitraphylline's exact 2D-constitutional stereoisomer family, as found in COCONUT.

In [ ]:

img_b = make_structure_grid(top10_distinct, f"{FIG_DIR}/fig8b_mitraphylline_distinct_analogues_grid.png")
img_b


**Tier B** (above): structurally distinct analogues -- these include known corynanthe-type alkaloid relatives (e.g. mitraphyllic acid, vineridine, vinerine), representing genuine candidates for analogue-based follow-up rather than stereoisomeric duplicates.

In [ ]:

def make_ranking_barplot(df, out_path, title_suffix):
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(df["rank"].astype(str) + ". " + df["identifier"],
                    df["tanimoto_to_mitraphylline"], color="#2b6f77")
    ax.invert_yaxis()
    ax.set_xlabel("Tanimoto Similarity to Mitraphylline (Morgan/ECFP4, r=2, 2048-bit)")
    ax.set_title(f"Top-10 {title_suffix} of Mitraphylline in COCONUT\n"
                 f"(ranked among n={n_rows:,} unique natural products)", fontsize=10, fontweight="bold")
    for bar, val in zip(bars, df["tanimoto_to_mitraphylline"]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center", fontsize=8)
    ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

make_ranking_barplot(top10, f"{FIG_DIR}/fig9_mitraphylline_similarity_ranking.png", "Nearest Neighbors")
make_ranking_barplot(top10_distinct, f"{FIG_DIR}/fig9b_mitraphylline_distinct_analogues_ranking.png",
                     "Structurally Distinct Analogues")


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(sims, bins=200, color="#8c8c8c", alpha=0.85, log=True)
for _, row in top10.iterrows():
    ax.axvline(row["tanimoto_to_mitraphylline"], color="#c44e52", linewidth=0.8, alpha=0.7)
ax.axvline(sim_mean, color="#4c72b0", linestyle="--", linewidth=1.5, label=f"Mean = {sim_mean:.3f}")
ax.set_xlabel("Tanimoto Similarity to Mitraphylline"); ax.set_ylabel("Count (log scale)")
ax.set_title(f"Distribution of Mitraphylline Similarity Across the Full COCONUT Set\n"
             f"(n={n_rows:,}; red lines = top-10 hits, all beyond the "
             f"{top10['percentile_in_full_distribution'].min():.2f}th percentile)",
             fontsize=10, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/fig10_mitraphylline_similarity_distribution.png", dpi=300, bbox_inches="tight")
plt.show()



## 6. Summary of Generated Artifacts

For traceability and reuse, every table and figure produced by this notebook is listed below with its output path.


In [ ]:

csv_outputs = sorted([f for f in os.listdir(CSV_DIR) if f.endswith(".csv")])
fig_outputs = sorted([f for f in os.listdir(FIG_DIR) if f.endswith(".png")])

print(f"CSV tables ({len(csv_outputs)}):")
for f in csv_outputs:
    size_kb = os.path.getsize(f"{CSV_DIR}/{f}") / 1024
    print(f"  - {f}  ({size_kb:,.0f} KB)")

print(f"\nFigures, 300 DPI PNG ({len(fig_outputs)}):")
for f in fig_outputs:
    size_kb = os.path.getsize(f"{FIG_DIR}/{f}") / 1024
    print(f"  - {f}  ({size_kb:,.0f} KB)")



## 7. Limitations & Reproducibility Notes

**Methodological limitations:**
- **Fingerprint stereochemistry blindness.** The Morgan/ECFP4 fingerprints used throughout (as generated by RDKit's default `MorganGenerator`) do not encode stereochemistry. This is precisely why mitraphylline's Tier-A "nearest neighbors" are exact-similarity stereoisomers rather than novel structures (Section 5) — a limitation we address directly by introducing the Tier-B distinct-analogue filter, but which would need chirality-aware fingerprints (`includeChirality=True`) or 3D shape-based methods (e.g. ROCS, USR) to resolve at the similarity-search level itself.
- **UMAP subsampling.** The 2D chemical-space visualization (Section 4) is computed on a 30,000-compound stratified subsample for computational tractability; the underlying PCA variance statistics, by contrast, use the complete ~700K-compound dataset. Results should be interpreted as representative rather than exhaustive at the visualization level.
- **NPClassifier annotation coverage.** ~6% of compounds lack an NPClassifier pathway assignment (grouped as "Unclassified" throughout); pathway-stratified statistics are computed only over annotated compounds.
- **COCONUT metadata heterogeneity.** Fields such as `organisms`, `cas`, and `dois` have substantial missingness (60–92%) reflecting upstream source-database heterogeneity; no analysis in this notebook depends on these fields.

**Reproducibility:**
- All random operations (stratified subsampling, UMAP) use a fixed seed (`SEED = 42`).
- Every heavy computational stage is idempotent and checkpointed/cached, so partial re-runs after interruption reproduce identical results to a full run.
- The full pipeline was validated end-to-end on a constrained 1-vCPU / 4 GB RAM machine, processing 715,822 raw COCONUT records to 705,603 cleaned, fingerprinted, analysis-ready compounds.
- Dependency versions are not hard-pinned in the installation cell above; for strict computational reproducibility, capture the environment with `pip freeze > requirements.txt` after a successful run and archive it alongside this notebook.

**Generalizability beyond mitraphylline:**
Sections 1–4 (cleaning, fingerprinting, chemical space mapping) are entirely reference-compound-agnostic and require no modification for other natural-product datasets. Section 5 requires only an InChIKey (or a SMILES string parsed to one) to re-target the analogue-prioritization workflow at any other reference compound of interest.



## References & Software Citations

- **COCONUT database:** Sorokina, M., Merseburger, P., Rajan, K., Yirik, M.A., Steinbeck, C. (2021). COCONUT online: Collection of Open Natural Products database. *Journal of Cheminformatics*, 13, 2.
- **RDKit:** RDKit: Open-source cheminformatics. https://www.rdkit.org
- **NPClassifier:** Kim, H.W., et al. (2021). NPClassifier: A Deep Neural Network-Based Structural Classification Tool for Natural Products. *Journal of Natural Products*, 84(11), 2795–2807.
- **QED (drug-likeness):** Bickerton, G.R., et al. (2012). Quantifying the chemical beauty of drugs. *Nature Chemistry*, 4, 90–98.
- **UMAP:** McInnes, L., Healy, J., Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. *arXiv:1802.03426*.
- **Morgan/ECFP fingerprints:** Rogers, D., Hahn, M. (2010). Extended-Connectivity Fingerprints. *J. Chem. Inf. Model.*, 50(5), 742–754.
- **Lipinski's Rule of Five:** Lipinski, C.A., et al. (2001). Experimental and computational approaches to estimate solubility and permeability. *Adv. Drug Deliv. Rev.*, 46(1–3), 3–26.
- **Veber's Rule:** Veber, D.F., et al. (2002). Molecular properties that influence the oral bioavailability of drug candidates. *J. Med. Chem.*, 45(12), 2615–2623.
- **Mitraphylline pharmacology (background):** consult primary literature on *Uncaria*/*Mitragyna* oxindole alkaloids for pharmacological context beyond the cheminformatic scope of this notebook.
